# Dispatcher Coupled

In [20]:
from pringles.simulator import Simulator
mySimulator = Simulator(cdpp_bin_path='bin/', user_models_dir='src/')

In [21]:
atomics = dict([(atomic.__name__, atomic) for atomic in mySimulator.atomic_registry.discovered_atomics])
Dispatcher = atomics['Dispatcher']

In [22]:
from pringles.models import Coupled 
a_dispatcher = Dispatcher("a_dispatcher", numberOfServers=5, server2="free")

print("Inport names: ", [port.name for port in a_dispatcher.inports])
print("Outport names: ", [port.name for port in a_dispatcher.outports])

Inport names:  ['newJob', 'jobDone', 'serverStatus']
Outport names:  ['requestJob', 'server0', 'server1', 'server2']


In [23]:
top_model = (Coupled(name='top', subcomponents=[a_dispatcher])
                .add_inport("newJob")
                .add_inport("jobDone")             
                .add_inport("serverStatus")
                .add_outport("requestJob")
                .add_outport("server0")
                .add_outport("server1")
                .add_outport("server2") 
                .add_coupling('newJob', a_dispatcher.get_port("newJob"))
                .add_coupling('jobDone', a_dispatcher.get_port('jobDone'))
                .add_coupling('serverStatus', a_dispatcher.get_port('serverStatus'))
             
                .add_coupling(a_dispatcher.get_port('requestJob'), "requestJob")
                .add_coupling(a_dispatcher.get_port('server0'), "server0")
                .add_coupling(a_dispatcher.get_port('server1'), "server1")
                .add_coupling(a_dispatcher.get_port('server2'), "server2")
            )
top_model

In [24]:
from pringles.simulator import Simulation, Event
from pringles.utils import VirtualTime

In [25]:
import os

working_dir = 'sim_results/'

# Create the directory structure if it does not exist
if not os.path.exists(working_dir):
    os.makedirs(working_dir)

In [26]:
sim_events = [
    Event(VirtualTime(0,0,20,0,0), top_model.get_port('serverStatus'), [float(2),float(0)]),
    Event(VirtualTime(0,0,25,0,0), top_model.get_port('newJob'), float(1)),
    Event(VirtualTime(0,0,35,0,0), top_model.get_port('serverStatus'), [float(3),float(1)]),
    Event(VirtualTime(0,0,40,0,0), top_model.get_port('serverStatus'), [float(1),float(1)]),
    Event(VirtualTime(0,0,42,0,0), top_model.get_port('newJob'), float(1))
]

a_simulation = Simulation(top_model = top_model, 
                          duration = VirtualTime.of_minutes(50), 
                          events=sim_events,
                          working_dir = working_dir
                         )

results = mySimulator.run_simulation(a_simulation)

In [27]:
print(results.get_process_output())

PCD++: A Tool to Implement n-Dimensional Cell-DEVS models
Version 3.0 - March 2003
Troccoli A., Rodriguez D., Wainer G., Barylko A., Beyoglonian J., Lopez A.
-----------------------------------------------------------------------------
PCD++ Extended States: An extended and improved version of CD++ for Cell-DEVS
Version 4.1.2 - December 2018
Santi L., Castro, R., Pimás, J.
-----------------------------------------------------------------------------
Discrete Event Simulation Lab
Departamento de Computación
Facultad de Ciencias Exactas y Naturales
Universidad de Buenos Aires, Argentina
-----------------------------------------------------------------------------
Compiled for standalone simulation


Loading models from sim_results/2024-03-28-155706-8d3fd2f8d7d9460b9de00741492804e0/top_model
Loading events from sim_results/2024-03-28-155706-8d3fd2f8d7d9460b9de00741492804e0/events
Running parallel simulation. Reading models partition from 
Model partition details output to: /dev/null*
Mess

In [28]:
display(results.output_df.head(100))

,time,port,value


In [29]:
print(results.logs_dfs.keys(),'\n\n')
display(results.logs_dfs['ParallelRoot'].head())

dict_keys(['a_dispatcher', 'top', 'ParallelRoot']) 




,0,1,message_type,time,model_origin,port,value,model_dest
